<a href="https://colab.research.google.com/github/narendrapatel6321-dotcom/sec-10k-crag/blob/main/notebooks/10k_pinecone_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SEC 10-K RAG Ingestion Pipeline
Downloads SEC 10-K filings, extracts structured XBRL financials and prose sections
(Risk Factors / MD&A), then builds a hybrid retrieval index: dense embeddings upserted
to Pinecone + a local BM25 sparse index.

## Install Dependencies

In [1]:
!pip install -q --upgrade \
    "numpy<2.1" \
    "pandas==2.2.2" \
    "requests==2.32.4" \
    edgartools \
    sec-edgar-downloader \
    langchain \
    langchain-core \
    langchain-community \
    langchain-pinecone \
    pinecone-client \
    langchain-huggingface \
    sentence-transformers \
    rank_bm25 \
    langgraph \
    langgraph-prebuilt \
    langgraph-sdk

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.9/60.9 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 65.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 68.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 461.3/461.3 kB 41.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 109.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 95.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 611.3/611.3 kB 49.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.4/155.4 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.0/76.0 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.2/46.2 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 587.6/587.6 kB 43.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69

## Mount Drive & Setup Paths

In [2]:
import os
from pathlib import Path
from google.colab import drive
from google.colab import userdata

drive.mount('/content/drive')

PROJECT_ROOT = Path('/content/drive/MyDrive/sec-10k-rag')
DATA_DIR = PROJECT_ROOT / 'data'
XBRL_DIR = DATA_DIR / 'xbrl_financials'
PROSE_DIR = DATA_DIR / 'prose_sections'
INDEX_DIR = DATA_DIR / 'index'

for directory in [DATA_DIR, XBRL_DIR, PROSE_DIR, INDEX_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

# SEC identity registration
COMPANY_NAME = userdata.get('SEC_COMPANY_NAME')
EMAIL = userdata.get('SEC_EMAIL')

TICKERS = ["C", "AAPL", "GOOGL", "GS", "MSFT"]
FILINGS_PER_TICKER = 3

print(f"Working directory: {PROJECT_ROOT}")

Mounted at /content/drive
Working directory: /content/drive/MyDrive/sec-10k-rag


## Raw Filing Download

In [3]:
from sec_edgar_downloader import Downloader

dl = Downloader(COMPANY_NAME, EMAIL, str(DATA_DIR))

print(">>> Checking / Downloading raw 10-K filings...")
for ticker in TICKERS:
    ticker_10k_dir = DATA_DIR / "sec-edgar-filings" / ticker / "10-K"

    # Check if filings are already cached
    if ticker_10k_dir.exists():
        existing = [d for d in ticker_10k_dir.iterdir() if d.is_dir()]
        if len(existing) >= FILINGS_PER_TICKER:
            print(f"  [CACHE] {ticker}: {len(existing)} filings present. Skipping download.")
            continue

    print(f"  [DOWNLOAD] Fetching {FILINGS_PER_TICKER} filings for {ticker}...")
    try:
        dl.get("10-K", ticker, limit=FILINGS_PER_TICKER, download_details=True)
        print(f"  [SUCCESS] {ticker} downloaded.")
    except Exception as e:
        print(f"  [ERROR] {ticker}: {e}")

>>> Checking / Downloading raw 10-K filings...
  [DOWNLOAD] Fetching 3 filings for C...
  [SUCCESS] C downloaded.
  [DOWNLOAD] Fetching 3 filings for AAPL...
  [SUCCESS] AAPL downloaded.
  [DOWNLOAD] Fetching 3 filings for GOOGL...
  [SUCCESS] GOOGL downloaded.
  [DOWNLOAD] Fetching 3 filings for GS...
  [SUCCESS] GS downloaded.
  [DOWNLOAD] Fetching 3 filings for MSFT...
  [SUCCESS] MSFT downloaded.


## Structured XBRL Financials Extraction

In [4]:
from edgar import Company, set_identity
from datetime import datetime
import pandas as pd
import pickle

set_identity(f"{COMPANY_NAME} {EMAIL}")

xbrl_records = {}
xbrl_failures = []

print(">>> Extracting structured XBRL statements...")

for ticker in TICKERS:
    company = Company(ticker)
    filings = company.get_filings(form="10-K", amendments=False).head(FILINGS_PER_TICKER)

    for filing in filings:
        acc_no = filing.accession_no
        key = f"{ticker}_{acc_no}"
        try:
            xbrl = filing.xbrl()
            fin = xbrl.statements

            income_stmt = fin.income_statement().to_dataframe()
            balance_sheet = fin.balance_sheet().to_dataframe()
            cash_flow = fin.cash_flow_statement().to_dataframe()

            por = getattr(filing, "period_of_report", None)
            if por:
                if isinstance(por, str):
                    por = datetime.strptime(por, "%Y-%m-%d")
                fiscal_year = por.year
            else:
                fiscal_year = filing.filing_date.year if filing.filing_date.month > 6 else filing.filing_date.year - 1

            xbrl_records[key] = {
                "ticker": ticker,
                "fiscal_year": fiscal_year,
                "accession": acc_no,
                "income_statement": income_stmt,
                "balance_sheet": balance_sheet,
                "cash_flow": cash_flow,
            }

            out_dir = XBRL_DIR / key
            out_dir.mkdir(exist_ok=True)
            income_stmt.to_csv(out_dir / "income_statement.csv", index=False)
            balance_sheet.to_csv(out_dir / "balance_sheet.csv", index=False)
            cash_flow.to_csv(out_dir / "cash_flow.csv", index=False)

            print(f"  [XBRL] {key} — FY{fiscal_year}")
        except Exception as e:
            print(f"  [ERROR] {key}: {e}")
            xbrl_failures.append((ticker, acc_no, str(e)))

with open(XBRL_DIR / "all_xbrl_records.pkl", "wb") as f:
    pickle.dump(xbrl_records, f)

print(f"\nTotal XBRL records saved: {len(xbrl_records)}")

>>> Extracting structured XBRL statements...
  [XBRL] C_0000831001-26-000011 — FY2025
  [XBRL] C_0000831001-25-000067 — FY2024
  [XBRL] C_0000831001-24-000033 — FY2023
  [XBRL] AAPL_0000320193-25-000079 — FY2025
  [XBRL] AAPL_0000320193-24-000123 — FY2024
  [XBRL] AAPL_0000320193-23-000106 — FY2023
  [XBRL] GOOGL_0001652044-26-000018 — FY2025
  [XBRL] GOOGL_0001652044-25-000014 — FY2024
  [XBRL] GOOGL_0001652044-24-000022 — FY2023
  [XBRL] GS_0000886982-26-000091 — FY2025
  [XBRL] GS_0000886982-25-000005 — FY2024
  [XBRL] GS_0000886982-24-000006 — FY2023
  [XBRL] MSFT_0001193125-26-323660 — FY2026
  [XBRL] MSFT_0000950170-25-100235 — FY2025
  [XBRL] MSFT_0000950170-24-087843 — FY2024

Total XBRL records saved: 15


## Prose Extraction (Item 1A & Item 7)

In [5]:
prose_records = {}
prose_failures = []

print(">>> Extracting MD&A and Risk Factors...")

for ticker in TICKERS:
    company = Company(ticker)
    filings = company.get_filings(form="10-K", amendments=False).head(FILINGS_PER_TICKER)

    for filing in filings:
        acc_no = filing.accession_no
        key = f"{ticker}_{acc_no}"
        try:
            tenk = filing.obj()
            risk_factors = tenk["Item 1A"]
            mdna = tenk["Item 7"]

            prose_records[key] = {
                "ticker": ticker,
                "accession": acc_no,
                "risk_factors": risk_factors,
                "mdna": mdna,
            }

            print(f"  [PROSE] {key}: Risk Factors={len(risk_factors) if risk_factors else 0} chars, MD&A={len(mdna) if mdna else 0} chars")

            if not risk_factors:
                prose_failures.append((ticker, acc_no, "risk_factors_missing"))
            if not mdna:
                prose_failures.append((ticker, acc_no, "mdna_missing"))

        except Exception as e:
            print(f"  [ERROR] {key}: {e}")
            prose_failures.append((ticker, acc_no, str(e)))

with open(PROSE_DIR / "all_prose_records.pkl", "wb") as f:
    pickle.dump(prose_records, f)

print(f"\nTotal Prose records saved: {len(prose_records)}")
print(f"Extraction issues: {prose_failures}")

>>> Extracting MD&A and Risk Factors...
  [PROSE] C_0000831001-26-000011: Risk Factors=88529 chars, MD&A=407746 chars
  [PROSE] C_0000831001-25-000067: Risk Factors=96990 chars, MD&A=446022 chars
  [PROSE] C_0000831001-24-000033: Risk Factors=100297 chars, MD&A=433098 chars
  [PROSE] AAPL_0000320193-25-000079: Risk Factors=68163 chars, MD&A=18018 chars
  [PROSE] AAPL_0000320193-24-000123: Risk Factors=68887 chars, MD&A=15358 chars
  [PROSE] AAPL_0000320193-23-000106: Risk Factors=67998 chars, MD&A=15509 chars
  [PROSE] GOOGL_0001652044-26-000018: Risk Factors=85311 chars, MD&A=52545 chars
  [PROSE] GOOGL_0001652044-25-000014: Risk Factors=83432 chars, MD&A=58046 chars
  [PROSE] GOOGL_0001652044-24-000022: Risk Factors=76171 chars, MD&A=59669 chars
  [PROSE] GS_0000886982-26-000091: Risk Factors=142138 chars, MD&A=310379 chars
  [PROSE] GS_0000886982-25-000005: Risk Factors=144556 chars, MD&A=294910 chars
  [PROSE] GS_0000886982-24-000006: Risk Factors=141927 chars, MD&A=291378 chars
  

## Chunking & Hybrid Index Creation (Pinecone + BM25)

In [6]:
!pip install -q langchain-text-splitters

In [7]:
# Fix for numpy/scipy version incompatibility

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_pinecone import PineconeVectorStore
from langchain_community.retrievers import BM25Retriever
from pinecone import Pinecone, ServerlessSpec
import pickle

PINECONE_API_KEY = userdata.get("PINECONE_API_KEY")
PINECONE_INDEX_NAME = "sec-10k-rag"
PINECONE_CLOUD = "aws"
PINECONE_REGION = "us-east-1"
EMBEDDING_DIM = 384  # BAAI/bge-small-en-v1.5 output dimension

In [8]:
print(">>> Loading prose sections for chunking...")
with open(PROSE_DIR / "all_prose_records.pkl", "rb") as f:
    prose_records = pickle.load(f)

# 1. Chunking
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", ".", " ", ""]
)

docs = []
for key, record in prose_records.items():
    for section_name in ["risk_factors", "mdna"]:
        content = record[section_name]
        if content:
            chunks = text_splitter.split_text(content)
            for i, chunk in enumerate(chunks):
                docs.append(Document(
                    page_content=chunk,
                    metadata={
                        "ticker": record["ticker"],
                        "accession": record["accession"],
                        "section": section_name,
                        "chunk_id": i
                    }
                ))

print(f"Total chunks created: {len(docs)}")

>>> Loading prose sections for chunking...
Total chunks created: 5613


In [9]:
# 2. Dense Vector Store (bge-small-en-v1.5) -> Pinecone
print("Generating dense embeddings and upserting into Pinecone...")
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",
    model_kwargs={"device": "cuda"},
    encode_kwargs={"normalize_embeddings": True}
)

pc = Pinecone(api_key=PINECONE_API_KEY)

if PINECONE_INDEX_NAME not in [idx["name"] for idx in pc.list_indexes()]:
    print(f"  [PINECONE] Creating index '{PINECONE_INDEX_NAME}'...")
    pc.create_index(
        name=PINECONE_INDEX_NAME,
        dimension=EMBEDDING_DIM,
        metric="cosine",
        spec=ServerlessSpec(cloud=PINECONE_CLOUD, region=PINECONE_REGION),
    )
else:
    print(f"  [PINECONE] Reusing existing index '{PINECONE_INDEX_NAME}'.")

pinecone_index = pc.Index(PINECONE_INDEX_NAME)

# Build the vector store connection ONCE, then upsert in batches with .add_documents()
vectorstore = PineconeVectorStore(index=pinecone_index, embedding=embeddings)

BATCH_SIZE = 100
for i in range(0, len(docs), BATCH_SIZE):
    batch = docs[i:i + BATCH_SIZE]
    vectorstore.add_documents(batch)
    print(f"  [PINECONE] Upserted batch {i // BATCH_SIZE + 1} ({len(batch)} chunks)")

print("All dense embeddings upserted to Pinecone.")

Generating dense embeddings and upserting into Pinecone...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

  [PINECONE] Creating index 'sec-10k-rag'...
  [PINECONE] Upserted batch 1 (100 chunks)
  [PINECONE] Upserted batch 2 (100 chunks)
  [PINECONE] Upserted batch 3 (100 chunks)
  [PINECONE] Upserted batch 4 (100 chunks)
  [PINECONE] Upserted batch 5 (100 chunks)
  [PINECONE] Upserted batch 6 (100 chunks)
  [PINECONE] Upserted batch 7 (100 chunks)
  [PINECONE] Upserted batch 8 (100 chunks)
  [PINECONE] Upserted batch 9 (100 chunks)
  [PINECONE] Upserted batch 10 (100 chunks)
  [PINECONE] Upserted batch 11 (100 chunks)
  [PINECONE] Upserted batch 12 (100 chunks)
  [PINECONE] Upserted batch 13 (100 chunks)
  [PINECONE] Upserted batch 14 (100 chunks)
  [PINECONE] Upserted batch 15 (100 chunks)
  [PINECONE] Upserted batch 16 (100 chunks)
  [PINECONE] Upserted batch 17 (100 chunks)
  [PINECONE] Upserted batch 18 (100 chunks)
  [PINECONE] Upserted batch 19 (100 chunks)
  [PINECONE] Upserted batch 20 (100 chunks)
  [PINECONE] Upserted batch 21 (100 chunks)
  [PINECONE] Upserted batch 22 (100 chun

In [10]:
# 3. Sparse Index (BM25)
print("Building and saving BM25 retriever...")
bm25_retriever = BM25Retriever.from_documents(docs)
with open(INDEX_DIR / "bm25_retriever.pkl", "wb") as f:
    pickle.dump(bm25_retriever, f)

print("BM25 index saved to disk.")
print("\n>>> Ingestion & Indexing Pipeline Complete!")

Building and saving BM25 retriever...
BM25 index saved to disk.

>>> Ingestion & Indexing Pipeline Complete!
